In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

import psutil
import math
import gc

import os, random
import posixpath
from pyhdas.frequency import spectrogram, add_db, energy, power_spectrum
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta

In [2]:
import warnings

# Suppress FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
def calculate_low_frequency_spectrum(
    strain,
    stride=16,
    low_frequency_cutoff=50,
    low_frequency_min=0.1,
):
    """Calculate low-frequency spectrum from strain signal

    Parameters
    ----------
    strain : xarray
        DAS strain measurement time series
    stride: integer, number of positions to calculate at once
        Using a higher stride will be faster on computers with a lot of RAM.
    low_frequency_cutoff : float, default 50
        Cutoff of the frequencies to discard when saving the low frequency
        outputs.
    low_frequency_min : float, default 0.1
        Minimal frequency for the power_spectrum of the low
        frequencies output

    Returns
    -------
    xarray
        low-frequency spectrum
    """

    all_spectra = []
    for j in range(0, len(strain.position), stride):
        low_freq_spectrum_ = power_spectrum(
            strain.isel(position=slice(j, j + stride)),
            min_frequency=low_frequency_min,
            keep_time_dim=True,
        )
        low_freq_spectrum_ = low_freq_spectrum_.sel(
            freq=slice(
                low_frequency_cutoff,
            )
        )
        all_spectra.append(low_freq_spectrum_)

    low_freq_spectrum = xr.concat(all_spectra, dim="position")

    return low_freq_spectrum

In [4]:
# select 200 random files from the directory
normal_data_dir = "data/19"
files = random.sample(os.listdir(normal_data_dir), 50)

In [5]:
def normal_low():

    for file in files:

        filepath = [posixpath.join(normal_data_dir, file)]

        # open file one by one
        ds_raw = concat_raw_data(filepath)

        # select locations 4210-4260
        poi = np.arange(1260, 6950, 10)

        ds_raw = ds_raw.sel(position=poi)
        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)
        
        fig,ax = plt.subplots(figsize=(15,6))
        ds_lowfreq_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
        ax.set_title(f"lowfrequency position spectrogram")
        
        # save in the normal data directory
        output_folder = Path(f"normal_low_freq_spectrograms")
        output_folder.mkdir(parents=True, exist_ok=True)

        plot_filename = f"{file[:-4]}_low_freq_spectrogram.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')

        plt.close(fig)
        
        print(f"Done with {file}")
        del ds_raw, ds_lowfreq_spect
        gc.collect()


In [ ]:
normal_low()

Done with 2021_02_19_08h55m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_20h24m41s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_15h59m40s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_00h24m38s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_08h42m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_18h41m40s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_07h06m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_03h30m38s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_02h53m38s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_23h34m41s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_09h14m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_01h46m38s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_09h51m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_01h34m38s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_23h04m41s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_13h17m40s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_19h44m41s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_21h06m41s_

In [11]:
# load the event table 
events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])

In [14]:
def event_level():
    # loop through event table 
    for _, event in events.iterrows():
        # load the event data
        start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]
        # select the locations: i chose the ones that are not in the poi list
        poi = np.arange(4200, 4310, 10)
        # if the duration of the event is more than 2 minutes, print the event_label, date and start and end time, and move on to the next row
        event_duration = (end - start).total_seconds()
        if event_duration > 120 or event_duration <= 1:
            continue
    
        # load the strain data based on the start and the end of the event
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)
        start = start.tz_localize("UTC")
        end = end.tz_localize("UTC")
        day = start.day

        dir_data = Path(fr"data/{day}")
        file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        ds_raw = concat_raw_data(file_list)
        start = start.tz_localize(None)
        end = end.tz_localize(None)

        # sound level 
        ds_raw = ds_raw.sel(time=slice(start, end), position=poi)
        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)

        fig,ax = plt.subplots(figsize=(15,6))
        ds_lowfreq_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
        ax.set_title(f"lowfrequency position spectrogram")

        # save the plot
        output_folder = Path(f"low_freq_plots/{label}")
        output_folder.mkdir(parents=True, exist_ok=True)
  
        plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_low_freq.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')

        plt.close(fig)
        
        print(f"Done with {start}, {end}, {label}")
        del ds_raw, ds_lowfreq_spect
        gc.collect()

In [ ]:
event_level()